# 03 · Analysis & Insights
**Payment Failure Intelligence & Revenue Optimization System**

Multi-layer analysis: Volume, Failure Diagnostics, Segmentation,
Time-Series, Statistical Testing, and Financial Impact.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')
os.makedirs('../outputs/charts', exist_ok=True)

# Style
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444', 'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0', 'xtick.color': '#aaa',
    'ytick.color': '#aaa', 'grid.color': '#2a2a3e',
    'grid.alpha': 0.6, 'font.family': 'sans-serif',
})
PALETTE = ['#6c63ff', '#ff6584', '#43e97b', '#f7971e', '#4facfe', '#a18cd1', '#fccb90']

df = pd.read_csv('../outputs/engineered_data.csv', parse_dates=['timestamp'])
daily = pd.read_csv('../outputs/daily_trends.csv', parse_dates=['date'])
print(f"Loaded: {df.shape}  |  Daily rows: {len(daily)}")

# ─── Helper ─────────────────────────────────────────────────────────────────
def save(name):
    plt.tight_layout()
    plt.savefig(f'../outputs/charts/{name}.png', dpi=150, bbox_inches='tight',
                facecolor=plt.rcParams['figure.facecolor'])
    plt.close()
    print(f"  ✓ Saved {name}.png")

## A · Volume Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Volume & Revenue Trends', fontsize=16, color='white', fontweight='bold', y=1.02)

monthly = df.groupby([df['timestamp'].dt.to_period('M')]).agg(
    total=('transaction_id', 'count'),
    revenue=('amount', 'sum'),
    failed=('is_failed', 'sum')
).reset_index()
monthly['period_str'] = monthly['timestamp'].astype(str)

ax = axes[0]
ax.bar(monthly['period_str'], monthly['total'], color='#6c63ff', alpha=0.85, label='Total')
ax.bar(monthly['period_str'], monthly['failed'], color='#ff6584', alpha=0.9, label='Failed')
ax.set_title('Monthly Transactions', color='white')
ax.set_xlabel('Month'); ax.set_ylabel('Transactions')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend()

ax = axes[1]
ax.plot(monthly['period_str'], monthly['revenue'] / 1e6, color='#43e97b', linewidth=2.5, marker='o', markersize=4)
ax.fill_between(range(len(monthly)), monthly['revenue'] / 1e6, alpha=0.2, color='#43e97b')
ax.set_title('Monthly Revenue Processed (₹ Lakhs)', color='white')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue (₹ Lakhs)')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['period_str'], rotation=45, ha='right', fontsize=7)

save('volume_analysis')

## B · Failure Diagnostics

In [ ]:
# --- B1: Failure rate by hour ---
hourly = df.groupby('hour').agg(total=('is_failed', 'count'), failed=('is_failed', 'sum'))
hourly['fail_rate'] = hourly['failed'] / hourly['total'] * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(hourly.index, hourly['fail_rate'], color='#ff6584', linewidth=2.5, marker='o', markersize=5)
ax.fill_between(hourly.index, hourly['fail_rate'], alpha=0.25, color='#ff6584')
ax.axhspan(0, 6, alpha=0.08, color='#6c63ff', label='Night window (0–5 AM)')
ax.set_title('Failure Rate by Hour of Day', fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Hour'); ax.set_ylabel('Failure Rate (%)')
ax.set_xticks(range(24))
ax.legend()
save('failure_by_hour')

# --- B2: By category ---
cat_stats = df.groupby('category').agg(
    total=('is_failed', 'count'), failed=('is_failed', 'sum')).reset_index()
cat_stats['fail_rate'] = cat_stats['failed'] / cat_stats['total'] * 100
cat_stats = cat_stats.sort_values('fail_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_stats['category'], cat_stats['fail_rate'],
               color=PALETTE[:len(cat_stats)], edgecolor='none')
for bar, val in zip(bars, cat_stats['fail_rate']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', color='white', fontsize=10)
ax.set_title('Failure Rate by Category', fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Failure Rate (%)')
save('failure_by_category')

# --- B3: By device ---
dev_stats = df.groupby('device_type').agg(
    total=('is_failed', 'count'), failed=('is_failed', 'sum')).reset_index()
dev_stats['fail_rate'] = dev_stats['failed'] / dev_stats['total'] * 100

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(dev_stats['device_type'], dev_stats['fail_rate'],
              color=['#6c63ff', '#43e97b', '#f7971e'], width=0.5, edgecolor='none')
for bar, val in zip(bars, dev_stats['fail_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', color='white', fontsize=11)
ax.set_title('Failure Rate by Device Type', fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Device'); ax.set_ylabel('Failure Rate (%)')
save('failure_by_device')

# --- B4: By region ---
reg_stats = df.groupby('region').agg(
    total=('is_failed', 'count'), failed=('is_failed', 'sum')).reset_index()
reg_stats['fail_rate'] = reg_stats['failed'] / reg_stats['total'] * 100
reg_stats = reg_stats.sort_values('fail_rate', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(reg_stats['region'], reg_stats['fail_rate'],
              color=PALETTE, width=0.5, edgecolor='none')
for bar, val in zip(bars, reg_stats['fail_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', color='white', fontsize=11)
ax.set_title('Failure Rate by Region', fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Failure Rate (%)')
save('failure_by_region')

## C · Time-Series Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax1 = axes[0]
ax1.plot(daily['date'], daily['total'], color='#6c63ff', linewidth=1.5, label='Total Transactions', alpha=0.7)
ax1.plot(daily['date'], daily['total'].rolling(7).mean(), color='white', linewidth=2, label='7-Day Rolling Avg')
ax1.set_title('Daily Transaction Volume', color='white', fontsize=13, fontweight='bold')
ax1.set_ylabel('Transactions'); ax1.legend()

ax2 = axes[1]
ax2.plot(daily['date'], daily['daily_fail_rate']*100, color='#ff6584', linewidth=1, alpha=0.5, label='Daily Rate')
ax2.plot(daily['date'], daily['rolling_7d_rate']*100, color='#f7971e', linewidth=2.5, label='7-Day Rolling Avg')
ax2.set_title('Daily Failure Rate (%)', color='white', fontsize=13, fontweight='bold')
ax2.set_ylabel('Failure Rate (%)'); ax2.legend()
ax2.set_xlabel('Date')

fig.suptitle('Transaction Trends Over Time', fontsize=16, color='white', fontweight='bold')
save('time_series')

## D · Segmentation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# New vs Returning
seg = df.groupby('user_type')['is_failed'].mean().reset_index()
seg['fail_rate'] = seg['is_failed'] * 100
axes[0].bar(seg['user_type'], seg['fail_rate'], color=['#6c63ff', '#ff6584'], width=0.4)
for i, (_, row) in enumerate(seg.iterrows()):
    axes[0].text(i, row['fail_rate'] + 0.2, f"{row['fail_rate']:.1f}%", ha='center', color='white', fontsize=13)
axes[0].set_title('Failure Rate: New vs Returning Users', color='white', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Failure Rate (%)')

# Amount bucket
amt = df.groupby('amount_bucket', observed=True)['is_failed'].mean().reset_index()
amt['fail_rate'] = amt['is_failed'] * 100
axes[1].bar(amt['amount_bucket'].astype(str), amt['fail_rate'],
            color=['#43e97b', '#f7971e', '#ff6584', '#a18cd1'], width=0.5)
for i, (_, row) in enumerate(amt.iterrows()):
    axes[1].text(i, row['fail_rate'] + 0.2, f"{row['fail_rate']:.1f}%", ha='center', color='white', fontsize=11)
axes[1].set_title('Failure Rate by Transaction Value', color='white', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

fig.suptitle('Segmentation Analysis', fontsize=16, color='white', fontweight='bold')
save('segmentation_analysis')

## E · Funnel Analysis

In [ ]:
total_initiated = len(df)
total_processed = int(total_initiated * 0.98)   # 2% drop at processing stage
total_success   = (df['status'] == 'Success').sum()

funnel_stages  = ['Initiated', 'Processed', 'Success']
funnel_values  = [total_initiated, total_processed, total_success]
funnel_colors  = ['#6c63ff', '#4facfe', '#43e97b']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(funnel_stages[::-1], funnel_values[::-1], color=funnel_colors[::-1], height=0.5)
for bar, val in zip(bars, funnel_values[::-1]):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({val/total_initiated*100:.1f}%)', va='center', color='white', fontsize=11)
ax.set_title('Transaction Funnel Analysis', fontsize=14, color='white', fontweight='bold')
ax.set_xlabel('Number of Transactions')
ax.set_xlim(0, total_initiated * 1.18)
save('funnel_analysis')

## F · Statistical Testing

In [ ]:
print("\n" + "="*60)
print("  STATISTICAL TESTING RESULTS")
print("="*60)

# --- Test 1: Chi-Square — Device Type vs Failure ---
contingency_device = pd.crosstab(df['device_type'], df['status'])
chi2_d, p_d, dof_d, _ = stats.chi2_contingency(contingency_device)
print(f"\n[Chi-Square] Device Type vs Failure Status")
print(f"  χ² = {chi2_d:.4f}  |  p-value = {p_d:.6f}  |  dof = {dof_d}")
print(f"  Result: {'SIGNIFICANT ✓' if p_d < 0.05 else 'NOT significant'} (α = 0.05)")
if p_d < 0.05:
    print("  → Device type significantly affects failure probability")

# --- Test 2: Chi-Square — Category vs Failure ---
contingency_cat = pd.crosstab(df['category'], df['status'])
chi2_c, p_c, dof_c, _ = stats.chi2_contingency(contingency_cat)
print(f"\n[Chi-Square] Category vs Failure Status")
print(f"  χ² = {chi2_c:.4f}  |  p-value = {p_c:.6f}  |  dof = {dof_c}")
print(f"  Result: {'SIGNIFICANT ✓' if p_c < 0.05 else 'NOT significant'} (α = 0.05)")

# --- Test 3: Mann-Whitney U — Amount (Success vs Failure) ---
success_amounts = df[df['status'] == 'Success']['amount']
failed_amounts  = df[df['status'] == 'Failed']['amount']
u_stat, p_mw = stats.mannwhitneyu(success_amounts, failed_amounts, alternative='two-sided')
print(f"\n[Mann-Whitney U] Transaction Amount: Success vs Failure")
print(f"  U = {u_stat:.0f}  |  p-value = {p_mw:.6f}")
print(f"  Result: {'SIGNIFICANT ✓' if p_mw < 0.05 else 'NOT significant'} (α = 0.05)")
if p_mw < 0.05:
    print(f"  → Median Success Amount: ₹{success_amounts.median():,.2f}")
    print(f"  → Median Failed  Amount: ₹{failed_amounts.median():,.2f}")

# --- Test 4: Night vs Day failure comparison ---
night_rate = df[df['is_night'] == 1]['is_failed'].mean()
day_rate   = df[df['is_night'] == 0]['is_failed'].mean()
contingency_night = pd.crosstab(df['is_night'], df['status'])
chi2_n, p_n, _, _ = stats.chi2_contingency(contingency_night)
print(f"\n[Chi-Square] Night (0–5 AM) vs Daytime Failure")
print(f"  Night failure rate : {night_rate:.2%}")
print(f"  Day   failure rate : {day_rate:.2%}")
print(f"  p-value = {p_n:.6f}  →  {'SIGNIFICANT ✓' if p_n < 0.05 else 'NOT significant'}")

## G · Financial Impact Analysis

In [ ]:
failed_df = df[df['status'] == 'Failed'].copy()

total_revenue_processed = df['amount'].sum()
total_revenue_lost      = failed_df['amount'].sum()
avg_failed_txn_value    = failed_df['amount'].mean()

print("\n" + "="*60)
print("  FINANCIAL IMPACT ANALYSIS  (INR)")
print("="*60)
print(f"\n  Total Revenue Processed : ₹{total_revenue_processed/1e7:,.2f} Cr")
print(f"  Total Revenue Lost      : ₹{total_revenue_lost/1e7:,.2f} Cr")
print(f"  Revenue Loss %          : {total_revenue_lost/total_revenue_processed*100:.2f}%")
print(f"  Avg Failed Txn Value    : ₹{avg_failed_txn_value:,.2f}")

monthly_loss = failed_df.groupby(
    failed_df['timestamp'].dt.to_period('M'))['amount'].sum().reset_index()
monthly_loss.columns = ['month', 'revenue_loss']
monthly_loss['month'] = monthly_loss['month'].astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(monthly_loss['month'], monthly_loss['revenue_loss']/1e5,
       color='#ff6584', alpha=0.9, edgecolor='none')
ax.plot(monthly_loss['month'], monthly_loss['revenue_loss'].rolling(3).mean()/1e5,
        color='white', linewidth=2.5, label='3-Month Rolling Avg')
ax.set_title('Monthly Revenue Loss Due to Failed Transactions (₹ Lakhs)', fontsize=13,
             color='white', fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Revenue Loss (₹ Lakhs)')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.legend()
save('revenue_loss')

## H · Export Summary & Insights

In [ ]:
total_txn          = len(df)
total_failed       = (df['status'] == 'Failed').sum()
overall_fail_rate  = total_failed / total_txn

top_fail_cat = cat_stats.sort_values('fail_rate', ascending=False).iloc[0]
top_fail_reg = reg_stats.iloc[0]
top_fail_dev = dev_stats.sort_values('fail_rate', ascending=False).iloc[0]
peak_hour    = hourly['fail_rate'].idxmax()

insights = pd.DataFrame([
    ('total_transactions',          total_txn),
    ('total_failed_transactions',   total_failed),
    ('overall_failure_rate_pct',    round(overall_fail_rate * 100, 2)),
    ('total_revenue_processed_inr', round(total_revenue_processed, 2)),
    ('total_revenue_lost_inr',      round(total_revenue_lost, 2)),
    ('revenue_loss_pct',            round(total_revenue_lost/total_revenue_processed*100, 2)),
    ('avg_failed_txn_value_inr',    round(avg_failed_txn_value, 2)),
    ('highest_failure_category',    top_fail_cat['category']),
    ('highest_failure_category_rate_pct', round(top_fail_cat['fail_rate'], 2)),
    ('highest_failure_region',      top_fail_reg['region']),
    ('highest_failure_region_rate_pct', round(top_fail_reg['fail_rate'], 2)),
    ('highest_failure_device',      top_fail_dev['device_type']),
    ('highest_failure_device_rate_pct', round(top_fail_dev['fail_rate'], 2)),
    ('peak_failure_hour',           peak_hour),
    ('peak_hour_failure_rate_pct',  round(hourly.loc[peak_hour, 'fail_rate'], 2)),
    ('night_failure_rate_pct',      round(night_rate * 100, 2)),
    ('day_failure_rate_pct',        round(day_rate * 100, 2)),
    ('chi2_device_pvalue',          round(p_d, 6)),
    ('chi2_category_pvalue',        round(p_c, 6)),
    ('mannwhitney_amount_pvalue',   round(p_mw, 6)),
], columns=['metric', 'value'])

insights.to_csv('../outputs/insights_summary.csv', index=False)
monthly_loss.to_csv('../outputs/monthly_revenue_loss.csv', index=False)

print("\n✓ Saved insights_summary.csv")
print("✓ Saved monthly_revenue_loss.csv")
print("\n=== ANALYSIS COMPLETE ===")
print(insights.to_string(index=False))